In [1]:
import numpy as np
from jaxtyping import Float
import pandas as pd
import matplotlib.pyplot as plt

# muutils
from muutils.jsonlines import jsonl_write, jsonl_load

# attention-motifs
from attention_motifs.bins import Bins
from attention_motifs.features.features import scalar_feature_table
from attention_motifs.features.hist_beta_fit import hist_beta_fit
from attention_motifs.util import prefix_dict
from attention_motifs.features.transition_tensor import tt_features
from attention_motifs.features.vec_features import vec_features
from attention_motifs.math.cos_sim import cosine_similarity_matrix
from attention_motifs.math.math import skew_lt

In [2]:
def gram_features(A: Float[np.ndarray, "n_ctx n_ctx"]) -> dict[str, float]:
	# dbg_tensor(A)
	return prefix_dict(
		hist_beta_fit(
			A.flatten(),
			bins=Bins(n_bins=32, start=0.0, stop=1.0),
		),
		prefix="beta_hist",
	)
	# TODO: mass as a function of distance from diagonal

In [3]:
def compute_scalar_features(
	A: Float[np.ndarray, "n_ctx n_ctx"],
) -> dict[str, float]:
	# dbg_tensor(A)
	A_log: Float[np.ndarray, "n_ctx n_ctx"] = np.nan_to_num(np.log(A + 1e-9), nan=-10)
	# dbg_tensor(A_log)

	A_skew: Float[np.ndarray, "n_ctx n_ctx"] = skew_lt(A)
	# dbg_tensor(A_skew)
	A_log_skew: Float[np.ndarray, "n_ctx n_ctx"] = skew_lt(A_log)

	return dict(
		# diagonal: standard features, fit diff to beta dist
		**prefix_dict(vec_features(A.diagonal()), prefix="diag"),
		# off-diagonal: standard features, fit diff to beta dist
		**prefix_dict(vec_features(A[:, 0]), prefix="first_tok"),
		# transition tensor: standard features, standard features on diff, linear envelope on transition time
		# 	TODO: standard features on decay rate
		# **prefix_dict(
		# 	tt_features(A),
		# 	prefix="markov_transition",
		# ),
		# # {log, raw} gram matrix of {rows, cols, rows of skewed}: beta fit hist
		# # 	TODO: fit fft in `gram_features`, but this is expensive
		# **prefix_dict(
		# 	gram_features(A @ A.T),
		# 	prefix=["gram", "row"],
		# ),
		# **prefix_dict(
		# 	gram_features(A.T @ A),
		# 	prefix=["gram", "col"],
		# ),
		# **prefix_dict(
		# 	gram_features(A_skew.T @ A_skew),
		# 	prefix=["gram", "skew"],
		# ),
		# **prefix_dict(
		# 	gram_features(cosine_similarity_matrix(A_log)),
		# 	prefix=["log", "gram", "row"],
		# ),
		# **prefix_dict(
		# 	gram_features(cosine_similarity_matrix(A_log, col=True)),
		# 	prefix=["log", "gram", "col"],
		# ),
		# **prefix_dict(
		# 	gram_features(cosine_similarity_matrix(A_log_skew)),
		# 	prefix=["log", "gram", "skew"],
		# ),
	)

In [4]:
df: pd.DataFrame = scalar_feature_table(
	features_func=compute_scalar_features,
	models="pythia-14m,gpt2-small".split(","),
)

models: ['pythia-14m', 'gpt2-small']
model: 'pythia-14m'
✔️  (0.00s) setting up paths                                                   
✔️  (0.00s) loading prompts                                                    
128 prompts loaded


prompts: 100%|██████████| 128/128 [00:13<00:00,  9.74it/s]

model: 'gpt2-small'
| (0.00s) setting up paths                                                     

✔️  (0.00s) setting up paths                                                   
✔️  (0.01s) loading prompts                                                    
142 prompts loaded


prompts: 100%|██████████| 142/142 [01:28<00:00,  1.60it/s]


In [2]:
path: str = "../data/scalar_features.jsonl.gz"


In [ ]:

jsonl_write(path, df.to_dict(orient="records"), use_gzip=True)

In [3]:
_temp_loaded = pd.DataFrame(jsonl_load(path))
df = _temp_loaded

In [7]:
df.describe()

,activation.layer,activation.head,feat.diag.mean,feat.diag.median,feat.diag.variance,feat.diag.std,feat.diag.skewness,feat.diag.kurtosis,feat.diag.entropy,feat.diag.L1_norm,...,feat.first_tok.L1_norm,feat.first_tok.L2_norm,feat.first_tok.rms,feat.first_tok.energy,feat.first_tok.zero_crossing_rate,feat.first_tok.autocorr_lag1,feat.first_tok.psd_total_power,feat.first_tok.linreg.slope,feat.first_tok.linreg.intercept,feat.first_tok.linreg.r2
count,23520.000000,23520.000000,23520.000000,2.352000e+04,23520.000000,23520.000000,23520.000000,23520.000000,23520.000000,23520.000000,...,23520.000000,23520.000000,23520.000000,23520.000000,23520.0,23520.000000,2.352000e+04,23520.000000,23520.000000,2.352000e+04
mean,5.108163,4.977551,0.092828,7.227908e-02,0.017892,0.123460,6.877460,61.872666,0.729998,0.092828,...,0.438840,0.051495,0.499840,32.999733,0.0,0.556244,3.486046e+00,-0.003029,0.568584,2.028350e-01
std,3.429815,3.512953,0.136546,1.584283e-01,0.022142,0.051471,3.175499,42.727725,0.721561,0.136546,...,0.271871,0.029614,0.243772,30.561891,0.0,0.266477,4.087028e+00,0.003121,0.281410,1.794614e-01
min,0.000000,0.000000,0.006212,3.548088e-42,0.003160,0.056210,-4.691017,-1.971993,0.033173,0.006212,...,0.003455,0.003448,0.058722,1.000000,0.0,-0.437013,5.527502e-10,-0.034680,0.012953,3.643010e-09
25%,2.000000,2.000000,0.029393,1.016021e-02,0.009069,0.095230,4.854771,28.171417,0.172037,0.029393,...,0.170496,0.025596,0.258190,6.492385,0.0,0.376276,5.787897e-01,-0.004002,0.335774,5.270615e-02
50%,5.000000,5.000000,0.049151,2.539381e-02,0.011267,0.106147,7.297240,59.123358,0.433121,0.049151,...,0.479345,0.051457,0.540049,27.446098,0.0,0.560581,2.191312e+00,-0.002333,0.603874,1.553435e-01
75%,8.000000,8.000000,0.096876,6.480474e-02,0.018320,0.135352,9.292452,90.346600,1.109145,0.096876,...,0.660685,0.069189,0.696422,49.615001,0.0,0.750562,5.193364e+00,-0.001076,0.805753,3.115373e-01
max,11.000000,11.000000,0.969806,9.942577e-01,0.247512,0.497506,16.359031,270.500506,3.293428,0.969806,...,0.999999,0.191864,0.999999,253.729767,0.0,0.999896,5.240953e+01,0.009146,1.238626,8.863901e-01


In [7]:
def plot_histograms_long(df: pd.DataFrame) -> None:
	"""Plot histograms for each feature with different models superimposed.

	This function assumes the DataFrame is in long format with columns:
	"model", "feat_name", "feat_val", and optionally "prompt", "layer", "head".

	# Parameters:
	 - `df : pd.DataFrame`
		 DataFrame containing the data.

	# Returns:
	 - `None`
		 Displays the histograms.
	"""
	features: list[str] = df["feat_name"].unique().tolist()
	models: list[str] = df["model"].unique().tolist()

	for feature in features:
		plt.figure()
		subset_feature: pd.DataFrame = df[df["feat_name"] == feature]
		for model in models:
			subset_model: pd.DataFrame = subset_feature[
				subset_feature["model"] == model
			]
			plt.hist(
				subset_model["feat_val"], bins=50, alpha=0.5, label=model, density=True
			)
		plt.xlabel(feature)
		plt.ylabel("Frequency")
		plt.title(f"Histogram of {feature} for different models")
		plt.legend()
		plt.show()


plot_histograms_long(df)

KeyError: 'feat_name'